# RisiKo RL — Training Bot

Notebook di training per il bot RL di RisiKo, da usare su **Colab Pro**.

**Workflow:**
1. Cella 1: monta Google Drive e clona il repo da GitHub
2. Cella 2: installa dipendenze RL
3. Cella 3: verifica che il simulatore funzioni
4. Cella 4: configura il training
5. Cella 5: lancia il training (può durare ore)
6. Cella 6: valuta il bot trainato
7. Cella 7 (opzionale): carica un modello esistente

**IMPORTANTE:** prima di iniziare, sostituisci `IL_TUO_USERNAME` e `il-tuo-repo` nella cella 1 con i tuoi dati GitHub.

## Cella 1: Setup Drive e clona repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/risiko-rl-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

# Clona il repo
%cd /content
!rm -rf risiko-rl
!git clone https://github.com/IL_TUO_USERNAME/il-tuo-repo.git risiko-rl
%cd risiko-rl

## Cella 2: Installa dipendenze

In [ ]:
!pip install -r requirements.txt -q
!pip install sb3-contrib -q
!pip install torch -q

import torch
import gymnasium as gym
from sb3_contrib import MaskablePPO
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponibile: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
print(f'Gymnasium: {gym.__version__}')

## Cella 3: Verifica simulatore

In [ ]:
for f in ['test_data', 'test_setup', 'test_motore', 'test_sdadata',
          'test_partita_completa', 'test_encoding', 'test_azioni', 'test_env']:
    print(f'=== {f} ===')
    !python tests/{f}.py 2>&1 | tail -3

## Cella 4: Configura training

In [ ]:
import sys
sys.path.insert(0, '/content/risiko-rl')

from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from stable_baselines3.common.callbacks import CheckpointCallback
import numpy as np
import torch

from risiko_env import RisikoEnv
from risiko_env import encoding as _encoding

# === STAGE A ATTIVO ===
# Aggiunge 12 feature di profilazione avversari (observation 318 -> 330).
# Baseline pre-Stage A: 27-29% WR. Obiettivo: rompere il soffitto.
_encoding.STAGE_A_ATTIVO = False
print(f'Stage A: {_encoding.STAGE_A_ATTIVO} (observation: {_encoding.get_dim_observation()} feature)')

def mask_fn(env):
    return env.get_action_mask()

def make_env(seed):
    def _init():
        # IMPORTANTE: ogni subprocess re-importa il modulo, quindi
        # impostiamo il flag QUI (non solo nel main process).
        from risiko_env import encoding as _encoding
        _encoding.STAGE_A_ATTIVO = False  # v8-selfplay: SELF-PLAY MINIMO (Stage D semplificato)
#
# Storia:
# - Baseline 500k (no Stage A): 29% WR vs random
# - Stage A v1 (4 feat): FALLITO 19%
# - Stage A2 (8 feat): non testato (cambio direzione)
#
# Cambio di prospettiva: il livello del bot e' limitato dal fatto che
# gioca SOLO contro random. Per scoprire strategie alte (cartina, balance
# of power, camuffamento obiettivo) servono avversari di livello.
#
# Self-play minimo:
# - Avversari = mix di copie precedenti del bot stesso (population)
# - 50% gen recenti (challenge), 50% mix random + gen vecchie (diversita')
# - Iperparametri: identici a v5-stable
# - 1M step per generazione (sopra 1M abbiamo visto degrado)
# - NO Stage A2 (semplificazione: prima validate self-play, poi Stage A2)
#
# Criterio successo (per ogni generazione):
# - gen N batte gen N-1 con WR > 50%
# - gen N batte random con WR > baseline 29%
# - emergenza patterns: cartina, attacco al leader, ecc. (test qualitativo)



## Cella 5: Lancia training

In [ ]:
# === CHECKPOINT + EARLY-STOPPING + BEST MODEL TRACKING ===
# Lezione del v4.1: il bot a 1M era migliore di quello a 3M.
# Quindi: salviamo checkpoint frequenti E un "best_model" basato su valutazione periodica.

from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback
from sb3_contrib.common.wrappers import ActionMasker

# Salva checkpoint regolari (ogni 50k step)
checkpoint_callback = CheckpointCallback(
    save_freq=50_000 // N_ENVS,
    save_path=CHECKPOINT_DIR_GEN,
    name_prefix='risiko_bot',
)

# Env di valutazione (1 sola partita per volta, con action masking)
def make_eval_env(seed):
    # Vedi nota sopra: imposta flag in subprocess.
    def _init():
        from risiko_env import encoding as _encoding
        _encoding.STAGE_A_ATTIVO = True  # v7-stagea2: Stage A attivo
        env = RisikoEnv(seed=seed)
        env = ActionMasker(env, mask_fn)
        return env
    return _init

# v5-stable: 4 env eval con seed diversi (10000, 20000, 30000, 40000)
# Cosi' la valutazione vede partite diverse, non sempre la stessa.
EVAL_SEEDS = [10000, 20000, 30000, 40000]
eval_env = SubprocVecEnv([make_eval_env(s) for s in EVAL_SEEDS])
eval_env = VecMonitor(eval_env)

# Salva automaticamente il MIGLIOR modello trovato durante il training
# (basato su reward medio in 20 partite di valutazione, ogni 100k step)
# v5-stable: EvalCallback robusta (no overfit al seed)
# - 20 episodi per misurazione affidabile (std > 0)
# - eval ogni 200k step (meno frequente ma piu' affidabile)
# - deterministic=False per esplorare seed diversi
# - VERBOSE: stampa anche +/- std per capire se la valutazione e' stabile
best_model_callback = MaskableEvalCallback(
    eval_env,
    best_model_save_path=CHECKPOINT_DIR_GEN + '/best/',
    log_path=CHECKPOINT_DIR_GEN + '/eval_logs/',
    eval_freq=200_000 // N_ENVS,  # ogni 200k step (era 100k)
    n_eval_episodes=20,
    deterministic=True,           # deterministic ok, ma su 20 partite con seed diversi
    verbose=1,
)

print('Callback configurate:')
print(f'  - Checkpoint ogni 50k step in {CHECKPOINT_DIR_GEN}')
print(f'  - Best model in {CHECKPOINT_DIR_GEN}/best/ (auto-aggiornato ogni 100k step)')
print(f'  - Valutazione su 20 partite, deterministica')

import time
inizio = time.time()
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[checkpoint_callback, best_model_callback],
    progress_bar=True,
)
durata = time.time() - inizio
print(f'\nTraining completato in {durata/60:.1f} minuti')

# Salva anche il modello finale (utile per debug)
model.save(f'{CHECKPOINT_DIR_GEN}/risiko_bot_finale')
print(f'Modello finale salvato.')
print(f'\nIMPORTANTE: per la valutazione USA il file in {CHECKPOINT_DIR_GEN}/best/best_model.zip')
print(f'Il modello finale (risiko_bot_finale.zip) potrebbe essere PEGGIORE del best.')

## Cella 6: Valuta il bot trainato

In [ ]:
from risiko_env import RisikoEnv
import numpy as np

def valuta_bot(model, n_partite=100, verbose=False):
    vittorie = 0
    rewards_totali = []
    durate_step = []

    for seed in range(n_partite):
        env = RisikoEnv(seed=seed)
        obs, info = env.reset()
        terminated = False
        truncated = False
        step_count = 0

        while not (terminated or truncated):
            mask = info['action_mask']
            action, _ = model.predict(obs, action_masks=mask, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))
            step_count += 1

        rewards_totali.append(reward)
        durate_step.append(step_count)
        if reward == 1.0:
            vittorie += 1

        if verbose and seed < 5:
            print(f'Partita {seed}: reward={reward}, step={step_count}, '
                  f'vincitore={info["vincitore"]}')

    win_rate = vittorie / n_partite
    reward_medio = np.mean(rewards_totali)
    step_medio = np.mean(durate_step)

    return {
        'win_rate': win_rate,
        'reward_medio': reward_medio,
        'step_medio': step_medio,
    }

risultati = valuta_bot(model, n_partite=100, verbose=True)
print()
print(f'Win rate: {risultati["win_rate"]*100:.1f}% (atteso ~25% per bot random)')
print(f'Reward medio: {risultati["reward_medio"]:.3f}')
print(f'Step medio: {risultati["step_medio"]:.0f}')

## Cella 7 (opzionale): Carica modello esistente

In [ ]:
import glob
checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/risiko_bot_*.zip'))
if checkpoints:
    print(f'Checkpoints disponibili:')
    for c in checkpoints:
        print(f'  {c}')
    model_loaded = MaskablePPO.load(checkpoints[-1], env=env)
    print(f'\nCaricato: {checkpoints[-1]}')
else:
    print('Nessun checkpoint trovato')

In [ ]:
# === COPIA BEST MODEL IN POPULATION ===
# Dopo training, copia il best_model.zip in population/genN.zip per la prossima generazione.
import shutil
src_best = f'{CHECKPOINT_DIR_GEN}/best/best_model.zip'
dst_pop = f'/content/drive/MyDrive/risiko-rl-checkpoints/population/gen{GEN_NUOVA}.zip'

if os.path.exists(src_best):
    shutil.copy(src_best, dst_pop)
    print(f'[OK] Best model gen{GEN_NUOVA} copiato in {dst_pop}')
    print(f'\nProssimo passo: valuta con')
    print(f'  python scripts/self_play.py eval --gen {GEN_NUOVA} --vs gen0 --n_partite 200')
    print(f'  python scripts/self_play.py eval --gen {GEN_NUOVA} --vs random --n_partite 200')
else:
    print(f'[!] best_model.zip non trovato. Usa risiko_bot_finale.zip come fallback?')